# Prueba de Concepto (PoC) V1.0 - TFM Fases 1 y 2

Este notebook implementa las dos primeras fases de la arquitectura híbrida GNN-HVA para la caracterización de fases topológicas, utilizando el Modelo de Ising en Campo Transversal (TFIM) 1D como sistema de prueba.

## Celda 1: Importaciones y Configuración Inicial

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse.linalg import eigsh
from scipy.optimize import minimize

# Importaciones modernas de Qiskit
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.circuit import QuantumCircuit, ParameterVector

# Fijar semilla para reproducibilidad en la optimización inicial
np.random.seed(42)

print("Librerías importadas correctamente. Entorno Qiskit listo.")

## Celda 2: FASE 1 - Generación del Ground Truth y Observables Locales
Construimos el Hamiltoniano TFIM, calculamos la energía exacta y medimos parámetros de orden locales para evadir la dependencia de métricas globales (resiliencia teórica ante *barren plateaus*).

In [ ]:
# Parámetros del Modelo de Juguete
N = 6          # Número de qubits
J = 1.0        # Fuerza de interacción
h_values = np.linspace(0.0, 2.0, 21) # Escaneo del campo magnético (21 puntos)

def build_tfim_hamiltonian(N, J, h):
    """Construye el Hamiltoniano TFIM usando la API moderna SparsePauliOp."""
    pauli_list = []
    
    # Interacción ZZ entre vecinos
    for i in range(N - 1):
        pauli_list.append(("ZZ", [i, i+1], -J))
        
    # Campo transversal X en cada qubit
    for i in range(N):
        pauli_list.append(("X", [i], -h))
        
    return SparsePauliOp.from_sparse_list(pauli_list, num_qubits=N)

# Almacenamiento de datos
exact_states = []
mag_X_list = []   # Magnetización local <X_0>
corr_ZZ_list = [] # Correlación local <Z_0 Z_1>

# Operadores locales para medir (usando la sintaxis moderna)
op_X0 = SparsePauliOp.from_sparse_list([("X", [0], 1.0)], num_qubits=N)
op_Z0Z1 = SparsePauliOp.from_sparse_list([("ZZ", [0, 1], 1.0)], num_qubits=N)

print("Calculando Diagonalización Exacta y Observables Locales...")

for h in h_values:
    H = build_tfim_hamiltonian(N, J, h)
    H_matrix = H.to_matrix(sparse=True)
    
    # Encontrar el autovalor más bajo (k=1) mediante fuerza bruta clásica
    evals, evecs = eigsh(H_matrix, k=1, which='SA')
    ground_state_vec = evecs[:, 0]
    exact_states.append(ground_state_vec)
    
    # Medir observables locales usando Statevector de Qiskit
    sv = Statevector(ground_state_vec)
    mag_X = sv.expectation_value(op_X0).real
    corr_ZZ = sv.expectation_value(op_Z0Z1).real
    
    mag_X_list.append(mag_X)
    corr_ZZ_list.append(corr_ZZ)

# Visualización de la Transición de Fase Cuántica
plt.figure(figsize=(8, 5))
plt.plot(h_values, corr_ZZ_list, 'o-', label=r'Correlación $\langle Z_0 Z_1 \rangle$')
plt.plot(h_values, mag_X_list, 's-', label=r'Magnetización $\langle X_0 \rangle$')
plt.axvline(x=1.0, color='red', linestyle='--', label='Punto Crítico $h=1$')
plt.xlabel('Campo Magnético (h)')
plt.ylabel('Valor Esperado Local')
plt.title('Transición de Fase en el Modelo de Ising 1D (Física Exacta)')
plt.legend()
plt.grid(True)
plt.show()

print("Fase 1 completada. Ground Truth generado.")

## Celda 3: FASE 2 (Parte A) - Diseño del Ansatz Físicamente Informado (HVA)
Restringimos la profundidad del circuito explícitamente para asegurar que la señal topológica sobreviva al ruido no unital en las fases posteriores.

In [ ]:
def create_hva_circuit(N, p):
    """
    Construye el Ansatz Variacional Hamiltoniano (HVA).
    Respetamos el límite de profundidad (p) para mitigar el ruido.
    """
    qc = QuantumCircuit(N)
    theta = ParameterVector('θ', 2 * p) # 2 parámetros por capa
    
    for layer in range(p):
        # Evolución ZZ (Interacción)
        for i in range(N - 1):
            qc.rzz(theta[layer * 2], i, i+1)
            
        qc.barrier() # Separador visual
        
        # Evolución X (Campo magnético transversal)
        for i in range(N):
            qc.rx(theta[layer * 2 + 1], i)
            
        if layer < p - 1:
            qc.barrier()
            
    return qc, theta

# Configuración del HVA superficial
p_layers = 2  # ¡Mantenemos el circuito MUY corto!
hva_qc, theta_params = create_hva_circuit(N, p_layers)

print(f"Circuito HVA creado con {hva_qc.num_parameters} parámetros libres.")
hva_qc.draw('mpl')

## Celda 4: FASE 2 (Parte B) - Compilación con Warm Start
Encontramos los parámetros óptimos. **Mejora QC:** Usamos el parámetro óptimo del paso anterior como semilla (`initial_guess`) para el siguiente. Al haber continuidad física en la función de onda, el optimizador converge en una fracción del tiempo.

In [ ]:
optimal_thetas = []
fidelities = []

print(f"Iniciando compilación hacia circuito HVA (Capas p={p_layers})...")

# MEJORA (Warm Start): Empezamos con una semilla aleatoria solo para el primer punto
current_guess = np.random.uniform(-np.pi, np.pi, hva_qc.num_parameters)

for idx, h in enumerate(h_values):
    target_state = Statevector(exact_states[idx])
    
    def cost_function(params):
        # Bind de parámetros y simulación rápida
        bound_qc = hva_qc.assign_parameters(params)
        sv_ansatz = Statevector(bound_qc)
        
        # Calculo de fidelidad matemáticamente exacto
        overlap = np.abs(np.vdot(target_state.data, sv_ansatz.data))**2
        return 1.0 - overlap

    # Optimización usando L-BFGS-B (usando el current_guess como semilla)
    res = minimize(cost_function, current_guess, method='L-BFGS-B', 
                   options={'maxiter': 1000, 'ftol': 1e-6})
    
    optimal_thetas.append(res.x)
    fidelities.append(1.0 - res.fun)
    
    # WARM START: Actualizamos la semilla para el próximo ciclo con el resultado óptimo actual
    current_guess = res.x

# Preparando Tensores para PyTorch (Fase 3)
X_data = np.array(h_values).reshape(-1, 1) # Entradas (h)
Y_data = np.array(optimal_thetas)          # Salidas target (thetas)

# Visualización de la Fidelidad de Compilación
plt.figure(figsize=(8, 4))
plt.plot(h_values, np.array(fidelities)*100, 'g^-', label='Fidelidad HVA')
plt.axvline(x=1.0, color='red', linestyle='--', label='Punto Crítico $h=1$')
plt.xlabel('Campo Magnético (h)')
plt.ylabel('Fidelidad (%)')
plt.title('Calidad de la Traducción al Circuito Cuántico')
plt.legend()
plt.grid(True)
plt.show()

print("\n--- RESULTADOS FASE 2 ---")
print(f"Fidelidad promedio en el dataset: {np.mean(fidelities)*100:.2f}%")
print(f"Fidelidad mínima (en el punto crítico): {np.min(fidelities)*100:.2f}%")
print("Dataset X_data e Y_data listo para entrenar la red neuronal en PyTorch.")